In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense

In [ ]:
df = pd.read_csv("/content/imdb-movies-dataset.csv")

In [ ]:
df.columns

Index(['Poster', 'Title', 'Year', 'Certificate', 'Duration (min)', 'Genre',
       'Rating', 'Metascore', 'Director', 'Cast', 'Votes', 'Description',
       'Review Count', 'Review Title', 'Review'],
      dtype='object')

In [ ]:
print(df.head())

                                              Poster  \
0  https://m.media-amazon.com/images/M/MV5BYWRkZj...   
1  https://m.media-amazon.com/images/M/MV5BZGI4NT...   
2  https://m.media-amazon.com/images/M/MV5BZjIyOT...   
3  https://m.media-amazon.com/images/M/MV5BMjA5Zj...   
4  https://m.media-amazon.com/images/M/MV5BNTk1MT...   

                               Title    Year Certificate  Duration (min)  \
0                    The Idea of You  2023.0           R           115.0   
1  Kingdom of the Planet of the Apes  2023.0       PG-13           145.0   
2                          Unfrosted  2023.0       PG-13            97.0   
3                       The Fall Guy  2023.0       PG-13           126.0   
4                        Challengers  2023.0           R           131.0   

                        Genre  Rating  Metascore           Director  \
0      Comedy, Drama, Romance     6.4       67.0  Michael Showalter   
1   Action, Adventure, Sci-Fi     7.3       66.0           Wes B

In [ ]:
df.isnull().sum()

,0
Poster,0
Title,0
Year,150
Certificate,2630
Duration (min),336
Genre,7
Rating,404
Metascore,2445
Director,5
Cast,39


In [ ]:
# Aa columns ma data almost missing che → drop karva best

# Drop karo:

# Poster
# Title
# Year
# Certificate
# Genre
# Director
# Cast
# Description
# Review Title

df.drop(columns=[
    'Poster','Title','Year','Certificate','Genre',
    'Director','Cast','Description','Review Title'
], inplace=True)

In [ ]:
# 2. Numerical columns → fill

# Columns:

# Duration (min)
# Rating
# Metascore
# Votes
# Review Count

# Strategy:

num_cols = ['Duration (min)', 'Rating', 'Metascore', 'Votes', 'Review Count']

for col in num_cols:
    df[col] = df[col].astype(str).str.replace(',', '')
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# label encoding
df['sentiment'] = df['Rating'].apply(lambda x: 1 if x >= 7 else 0)

In [ ]:
df.sentiment

,sentiment
0,0
1,1
2,0
3,1
4,1
...,...
9995,0
9996,1
9997,0
9998,0


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['Review'], df['sentiment'], test_size=0.2, random_state=42
)

Text Preprocessing

In [ ]:
# Tokenization
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train.fillna(''))

In [ ]:
# Convert text → sequence
X_train_seq = tokenizer.texts_to_sequences(X_train.fillna(''))
X_test_seq  = tokenizer.texts_to_sequences(X_test.fillna(''))

In [ ]:
# 3. Padding (same length)
X_train_pad = pad_sequences(X_train_seq, maxlen=200)
X_test_pad  = pad_sequences(X_test_seq, maxlen=200)

Build Models

In [ ]:
# RNN Model
model_rnn = Sequential()
model_rnn.add(Embedding(input_dim=5000, output_dim=64, input_length=200))
model_rnn.add(SimpleRNN(64))
model_rnn.add(Dense(1, activation='sigmoid'))

model_rnn.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# LSTM Model
model_lstm = Sequential()
model_lstm.add(Embedding(5000, 64, input_length=200))
model_lstm.add(LSTM(64))
model_lstm.add(Dense(1, activation='sigmoid'))

model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# GRU Model
model_gru = Sequential()
model_gru.add(Embedding(5000, 64, input_length=200))
model_gru.add(GRU(64))
model_gru.add(Dense(1, activation='sigmoid'))

model_gru.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train Models
history_rnn = model_rnn.fit(X_train_pad, y_train, epochs=3, batch_size=64, validation_split=0.2)

history_lstm = model_lstm.fit(X_train_pad, y_train, epochs=3, batch_size=64, validation_split=0.2)

history_gru = model_gru.fit(X_train_pad, y_train, epochs=3, batch_size=64, validation_split=0.2)

Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.6741 - loss: 0.6298 - val_accuracy: 0.6756 - val_loss: 0.6222
Epoch 2/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7306 - loss: 0.5318 - val_accuracy: 0.6525 - val_loss: 0.6591
Epoch 3/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9142 - loss: 0.2623 - val_accuracy: 0.6363 - val_loss: 0.7903
Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.6762 - loss: 0.6156 - val_accuracy: 0.6894 - val_loss: 0.5770
Epoch 2/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7533 - loss: 0.5194 - val_accuracy: 0.7188 - val_loss: 0.5413
Epoch 3/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8163 - loss: 0.4263 - val_accuracy: 0.7256 - val_loss: 0.5853
Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.6748 - loss: 0.6259 - val_accuracy: 0.6794 - val_loss: 0.6066
Epoch 2/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7219 - loss: 0.5443 - val_accuracy: 0.

In [ ]:
# Evaluate
rnn_acc  = model_rnn.evaluate(X_test_pad, y_test)[1]
lstm_acc = model_lstm.evaluate(X_test_pad, y_test)[1]
gru_acc  = model_gru.evaluate(X_test_pad, y_test)[1]

print("RNN Accuracy :", rnn_acc)
print("LSTM Accuracy:", lstm_acc)
print("GRU Accuracy :", gru_acc)

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.6370 - loss: 0.7840
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7055 - loss: 0.6286
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6740 - loss: 0.6620
RNN Accuracy : 0.6370000243186951
LSTM Accuracy: 0.7055000066757202
GRU Accuracy : 0.6740000247955322
